# Qwen3.5-2B Commercial Campaign SFT - Solo LLM

Notebook dedicado solo al fine-tuning del LLM para generar propuestas comerciales automotrices en JSON.

Este notebook **no** incluye Diffusers, SDXL, DreamBooth, LoRA visual ni generacion de imagenes. Usa el dataset corregido de 300 ejemplos:

```text
data/commercial_campaing_sft/commercial_campaign_sft_corrected_300.json
```

Flujo:

1. Validar GPU/runtime.
2. Instalar dependencias de Unsloth, TRL y Transformers.
3. Cargar y validar el JSON SFT.
4. Separar train/eval de forma reproducible.
5. Ejecutar baseline del modelo no tuneado.
6. Fine-tunear Qwen3.5-2B con LoRA/QLoRA.
7. Guardar adapter.
8. Comparar base vs fine-tuned con metricas estructurales y tabla lado a lado.


## 0. Runtime y GPU

Activa GPU antes de ejecutar entrenamiento. Para Colab T4 se usa un perfil conservador: `max_seq_length=1024`, batch size 1, acumulacion 8, rank LoRA 8 y una epoca.


In [ ]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM total: {props.total_memory / 1024**3:.2f} GB")
else:
    print("No se detecto GPU. Puedes validar datos, pero activa GPU para entrenar/evaluar modelos.")


## 1. Instalar dependencias

En Colab/Kaggle puede tomar varios minutos. Despues de instalar dependencias, si el runtime ya habia importado `torch`, `transformers` o `PIL`, reinicia el runtime una vez y continua desde la seccion 2.


In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    !pip install -q --upgrade pip
    !pip install -q --upgrade --no-cache-dir "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q --upgrade --no-cache-dir datasets trl peft accelerate transformers safetensors pandas scikit-learn matplotlib
    print("Dependencias instaladas. Si Colab ya habia importado torch/transformers, reinicia el runtime una vez.")
    !pip check


## 2. Imports, rutas y configuracion

El notebook intenta detectar el root del proyecto en local, Colab o Kaggle. Si tu repo queda en otra ruta, edita `PROJECT_ROOT` manualmente.


In [ ]:
import gc
import importlib
import json
import os
import random
import re
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path("/content/dmc-tp3-gen-multimodal"),
    Path("/kaggle/working/dmc-tp3-gen-multimodal"),
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data").exists()), Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data"
SFT_DIR = DATA_ROOT / "commercial_campaing_sft"
SFT_JSON_PATH = SFT_DIR / "commercial_campaign_sft_corrected_300.json"
SFT_TRAIN_PATH = SFT_DIR / "train_qwen35.json"
SFT_EVAL_PATH = SFT_DIR / "eval_qwen35.json"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
LLM_OUTPUT_DIR = OUTPUT_ROOT / "qwen35-commercial-lora"
EVAL_OUTPUT_DIR = OUTPUT_ROOT / "evaluation"
for path in [SFT_DIR, LLM_OUTPUT_DIR, EVAL_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LLM_BASE_MODEL = "unsloth/Qwen3.5-2B"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True

REQUIRE_MIN_SFT_EXAMPLES = 300
TRAIN_FRACTION = 0.80

RUN_BASELINE_EVAL = False
RUN_LLM_TRAINING = False
RUN_FINETUNED_EVAL = False
BUILD_COMPARISON_REPORT = True

MAX_EVAL_EXAMPLES = 5
EVAL_MAX_NEW_TOKENS = 200

print({
    "project_root": str(PROJECT_ROOT),
    "sft_json": str(SFT_JSON_PATH),
    "base_model": LLM_BASE_MODEL,
    "adapter_output": str(LLM_OUTPUT_DIR),
    "cuda": torch.cuda.is_available(),
})

for pkg in ["torch", "transformers", "datasets", "peft", "trl", "unsloth"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'installed')}")
    except Exception as exc:
        print(f"{pkg}: no disponible ({exc})")


## 3. Contrato del dataset SFT

Cada ejemplo debe tener `instruction`, `input` y `output`. El `output` esperado contiene 8 campos comerciales. `negative_prompt` no es obligatorio porque este notebook solo entrena el LLM textual.


In [ ]:
EXPECTED_SFT_OUTPUT_FIELDS = {
    "strategy",
    "recommended_channel",
    "channel_rationale",
    "channel_plan",
    "ad_copy",
    "image_prompt",
    "kpis",
    "business_note",
}

EXAMPLE_SFT_RECORD = {
    "instruction": "Actua como estratega publicitario para una concesionaria de vehiculos. Genera una propuesta de campana en JSON.",
    "input": "Objetivo: Lead Generation | Vehiculo: SUV hibrida | Rango: mid-range | Audiencia: Families 35-44 | Sector cliente: familias urbanas | Canal historico: Instagram | Ciudad: Miami | Idioma: Spanish | Duracion: 30 Days | Promocion: test drive + financiamiento | ROI: 2.10 | Conversion rate: 0.08 | Engagement: 9",
    "output": {
        "strategy": "Promover seguridad, espacio familiar y ahorro de combustible, cerrando con una invitacion a test drive.",
        "recommended_channel": "Instagram",
        "channel_rationale": "Instagram es visual, permite formularios de leads y encaja con audiencias familiares urbanas.",
        "channel_plan": "Usar Instagram para awareness visual y formularios de lead generation; reforzar con remarketing en Meta Ads.",
        "ad_copy": "Tu familia merece mas espacio, tecnologia y eficiencia. Agenda hoy tu test drive.",
        "image_prompt": "REALCARMODEL real car model in a Spanish Instagram ad for a mid-range hybrid SUV dealership campaign targeting urban families in Miami, premium automotive commercial photography, clear space for headline, no readable text",
        "kpis": ["Leads", "Cost per Lead", "Test Drive Bookings", "Conversion Rate", "ROI"],
        "business_note": "Priorizar leads calificados y medir reservas de test drive antes de escalar presupuesto."
    }
}

print("Campos esperados:", sorted(EXPECTED_SFT_OUTPUT_FIELDS))
print("Ubicacion esperada:", SFT_JSON_PATH)
print(json.dumps(EXAMPLE_SFT_RECORD, indent=2, ensure_ascii=False))


## 4. Cargar y validar JSON SFT

Valida que el dataset tenga 300 ejemplos y que cada salida incluya los 8 campos esperados.


In [ ]:
def load_sft_records(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}. Copia el JSON corregido de 300 ejemplos en esa ruta.")

    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, list):
        raise ValueError("El dataset SFT debe ser una lista JSON de objetos.")

    validated = []
    errors = []
    for idx, item in enumerate(data):
        if not isinstance(item, dict):
            errors.append((idx, "record is not an object"))
            continue
        for key in ["instruction", "input", "output"]:
            if key not in item:
                errors.append((idx, f"missing {key}"))
        output = item.get("output")
        if isinstance(output, str):
            try:
                output = json.loads(output)
                item = {**item, "output": output}
            except Exception:
                errors.append((idx, "output string is not valid JSON"))
        if not isinstance(item.get("output"), dict):
            errors.append((idx, "output is not an object"))
            continue
        missing = EXPECTED_SFT_OUTPUT_FIELDS - set(item["output"].keys())
        if missing:
            errors.append((idx, f"missing output fields: {sorted(missing)}"))
        validated.append(item)

    if errors:
        print("Errores de validacion encontrados, mostrando maximo 10:")
        for err in errors[:10]:
            print(err)
        raise ValueError(f"Dataset SFT invalido: {len(errors)} errores.")

    if len(validated) < REQUIRE_MIN_SFT_EXAMPLES:
        raise ValueError(f"Se esperaban {REQUIRE_MIN_SFT_EXAMPLES} ejemplos, se encontraron {len(validated)}.")

    print(f"Ejemplos SFT cargados: {len(validated)}")
    return validated

sft_records = load_sft_records(SFT_JSON_PATH)
display(pd.DataFrame([{
    "instruction": r["instruction"][:80],
    "input": r["input"][:140],
    "output_fields": sorted(r["output"].keys()),
} for r in sft_records[:5]]))


## 5. Split train/eval y formato instruct

El split se guarda para reproducibilidad. El texto de entrenamiento usa formato instruct simple: instruction, input y response JSON.


In [ ]:
def deterministic_split(records, train_fraction=0.8, seed=3407):
    records = list(records)
    rng = random.Random(seed)
    rng.shuffle(records)
    split_idx = int(len(records) * train_fraction)
    return records[:split_idx], records[split_idx:]

def format_sft_text(record):
    response = json.dumps(record["output"], ensure_ascii=False)
    return (
        "### Instruction:\n"
        f"{record['instruction']}\n\n"
        "### Input:\n"
        f"{record['input']}\n\n"
        "### Response:\n"
        f"{response}"
    )

def build_generation_prompt(record):
    return (
        "### Instruction:\n"
        f"{record['instruction']}\n\n"
        "### Input:\n"
        f"{record['input']}\n\n"
        "### Response:\n"
    )

train_records, eval_records = deterministic_split(sft_records, TRAIN_FRACTION, SEED)
SFT_TRAIN_PATH.write_text(json.dumps(train_records, ensure_ascii=False, indent=2), encoding="utf-8")
SFT_EVAL_PATH.write_text(json.dumps(eval_records, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Train: {len(train_records)} | Eval: {len(eval_records)}")
print("Train path:", SFT_TRAIN_PATH)
print("Eval path:", SFT_EVAL_PATH)
print(format_sft_text(train_records[0])[:1400])


## 6. Helpers de inferencia y metricas

Las metricas son simples y pensadas para comparar estructura, no para reemplazar revision humana.

- `json_valid`: la respuesta se puede parsear como JSON.
- `field_coverage`: porcentaje de campos obligatorios presentes.
- `recommended_channel_match`: coincidencia exacta del canal recomendado.
- `kpi_jaccard`: overlap entre KPIs esperados y generados.
- `text_jaccard_vs_expected`: similitud textual simple contra la salida esperada.
- `image_prompt_present`: presencia de `image_prompt` no vacio.
- `latency_sec`: tiempo de generacion.


In [ ]:
def extract_json_object(text):
    if isinstance(text, dict):
        return text
    if not isinstance(text, str):
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

def tokens_for_jaccard(value):
    return set(re.findall(r"\w+", str(value).lower()))

def jaccard(a, b):
    aw = tokens_for_jaccard(a)
    bw = tokens_for_jaccard(b)
    if not aw and not bw:
        return 1.0
    return len(aw & bw) / max(1, len(aw | bw))

def list_jaccard(a, b):
    aset = {str(x).strip().lower() for x in (a or [])}
    bset = {str(x).strip().lower() for x in (b or [])}
    if not aset and not bset:
        return 1.0
    return len(aset & bset) / max(1, len(aset | bset))

def score_generated_record(item):
    parsed = extract_json_object(item.get("generated", ""))
    expected = item.get("expected", {})
    json_valid = parsed is not None
    if not parsed:
        parsed = {}
    expected_channel = str(expected.get("recommended_channel", "")).strip().lower()
    generated_channel = str(parsed.get("recommended_channel", "")).strip().lower()
    return {
        "example_id": item.get("example_id"),
        "model": item.get("model"),
        "json_valid": json_valid,
        "field_coverage": len(EXPECTED_SFT_OUTPUT_FIELDS & set(parsed.keys())) / len(EXPECTED_SFT_OUTPUT_FIELDS),
        "recommended_channel_match": bool(expected_channel and generated_channel and expected_channel == generated_channel),
        "kpi_jaccard": list_jaccard(parsed.get("kpis"), expected.get("kpis")),
        "text_jaccard_vs_expected": jaccard(parsed, expected),
        "image_prompt_present": bool(str(parsed.get("image_prompt", "")).strip()),
        "latency_sec": item.get("latency_sec"),
    }

def score_llm_outputs(generated_records):
    return pd.DataFrame([score_generated_record(item) for item in generated_records])

print("Helpers de metricas listos.")


## 7. Carga de Qwen3.5-2B y generacion secuencial

Qwen3.5-2B se carga con `FastModel`, no con fallback automatico. Si falla la carga, el notebook muestra el error y recomienda revisar runtime/dependencias.


In [ ]:
def load_qwen35_for_generation(model_name_or_path):
    try:
        from unsloth import FastModel
    except Exception as exc:
        raise RuntimeError("No se pudo importar FastModel desde Unsloth. Revisa instalacion de dependencias.") from exc

    try:
        model, tokenizer = FastModel.from_pretrained(
            model_name=str(model_name_or_path),
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=LOAD_IN_4BIT,
        )
    except Exception as exc:
        raise RuntimeError(
            f"No se pudo cargar {model_name_or_path} con FastModel. "
            "Qwen3.5-2B requiere un runtime compatible con Unsloth/FastModel; "
            "no se cambiara automaticamente de modelo."
        ) from exc

    if hasattr(FastModel, "for_inference"):
        FastModel.for_inference(model)
    if getattr(tokenizer, "pad_token_id", None) is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

def cleanup_model(model=None, tokenizer=None):
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def generate_text(model, tokenizer, prompt, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors="pt")
    target_device = next(model.parameters()).device
    inputs = {key: value.to(target_device) for key, value in inputs.items()}
    prompt_tokens = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = outputs[0][prompt_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

def run_generation_phase(model_label, model_name_or_path, records, output_path):
    model, tokenizer = load_qwen35_for_generation(model_name_or_path)
    outputs = []
    try:
        for idx, record in enumerate(records):
            prompt = build_generation_prompt(record)
            start = time.perf_counter()
            generated = generate_text(model, tokenizer, prompt, max_new_tokens=EVAL_MAX_NEW_TOKENS)
            latency = time.perf_counter() - start
            outputs.append({
                "example_id": idx,
                "model": model_label,
                "generated": generated,
                "expected": record["output"],
                "latency_sec": latency,
            })
    finally:
        cleanup_model(model, tokenizer)

    output_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{model_label}: {len(outputs)} outputs guardados en {output_path}")
    return outputs

print("Helpers de generacion listos.")


## 8. Baseline del modelo no tuneado

Ejecuta esta seccion antes del entrenamiento para guardar respuestas del modelo base.


In [ ]:
BASE_OUTPUT_PATH = EVAL_OUTPUT_DIR / "qwen35_base_outputs.json"

if RUN_BASELINE_EVAL:
    if not torch.cuda.is_available():
        raise RuntimeError("Se recomienda GPU para evaluar Qwen3.5-2B.")
    eval_subset = eval_records[:MAX_EVAL_EXAMPLES]
    base_outputs = run_generation_phase("base", LLM_BASE_MODEL, eval_subset, BASE_OUTPUT_PATH)
    base_metrics = score_llm_outputs(base_outputs)
    display(base_metrics)
else:
    print("RUN_BASELINE_EVAL=False. Se omite baseline del modelo base.")
    print("Output esperado:", BASE_OUTPUT_PATH)


## 9. Fine-tuning LoRA/QLoRA con Qwen3.5-2B

Esta celda entrena adapters LoRA sobre atencion completa + MLP y guarda el resultado en `outputs/qwen35-commercial-lora/`.


In [ ]:
if RUN_LLM_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Se requiere GPU para entrenar Qwen3.5-2B con LoRA.")

    from datasets import Dataset
    from unsloth import FastModel
    from trl import SFTConfig, SFTTrainer

    model, tokenizer = FastModel.from_pretrained(
        model_name=LLM_BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
    )

    model = FastModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    train_dataset = Dataset.from_list([{"text": format_sft_text(r)} for r in train_records])
    eval_dataset = Dataset.from_list([{"text": format_sft_text(r)} for r in eval_records]) if eval_records else None

    training_args = SFTConfig(
        output_dir=str(LLM_OUTPUT_DIR),
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        warmup_ratio=0.03,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch" if eval_dataset is not None else "no",
        report_to="none",
    )

    trainer_kwargs = dict(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
    )
    try:
        trainer = SFTTrainer(**trainer_kwargs, processing_class=tokenizer)
    except TypeError:
        trainer = SFTTrainer(**trainer_kwargs, tokenizer=tokenizer)

    train_result = trainer.train()
    trainer.save_model(str(LLM_OUTPUT_DIR))
    tokenizer.save_pretrained(str(LLM_OUTPUT_DIR))
    print("Adapter Qwen3.5 LoRA guardado en:", LLM_OUTPUT_DIR)
    print(train_result)
    cleanup_model(model, tokenizer)
else:
    print("RUN_LLM_TRAINING=False. Se omite entrenamiento.")
    print("Adapter esperado:", LLM_OUTPUT_DIR)


## 10. Evaluacion del modelo fine-tuned

Recomendacion: despues de entrenar y guardar el adapter, reinicia el runtime y ejecuta desde imports hasta esta celda con `RUN_FINETUNED_EVAL=True`.


In [ ]:
FINETUNED_OUTPUT_PATH = EVAL_OUTPUT_DIR / "qwen35_finetuned_outputs.json"

if RUN_FINETUNED_EVAL:
    if not any(LLM_OUTPUT_DIR.glob("*")):
        raise RuntimeError(f"No se encontro adapter en {LLM_OUTPUT_DIR}. Ejecuta entrenamiento primero.")
    if not torch.cuda.is_available():
        raise RuntimeError("Se recomienda GPU para evaluar el adapter fine-tuned.")
    eval_subset = eval_records[:MAX_EVAL_EXAMPLES]
    ft_outputs = run_generation_phase("fine_tuned", LLM_OUTPUT_DIR, eval_subset, FINETUNED_OUTPUT_PATH)
    ft_metrics = score_llm_outputs(ft_outputs)
    display(ft_metrics)
else:
    print("RUN_FINETUNED_EVAL=False. Se omite evaluacion fine-tuned.")
    print("Output esperado:", FINETUNED_OUTPUT_PATH)


## 11. Reporte comparativo base vs fine-tuned

Esta seccion combina los outputs guardados y produce metricas agregadas y una tabla lado a lado.


In [ ]:
METRICS_PATH = EVAL_OUTPUT_DIR / "qwen35_llm_metrics.csv"
COMPARISON_TABLE_PATH = EVAL_OUTPUT_DIR / "qwen35_comparison_table.csv"

if BUILD_COMPARISON_REPORT:
    all_outputs = []
    for path in [BASE_OUTPUT_PATH, FINETUNED_OUTPUT_PATH]:
        if path.exists():
            all_outputs.extend(json.loads(path.read_text(encoding="utf-8")))
        else:
            print("No existe aun:", path)

    if not all_outputs:
        print("No hay outputs para comparar. Ejecuta baseline y/o fine-tuned eval primero.")
    else:
        metrics_df = score_llm_outputs(all_outputs)
        metrics_df.to_csv(METRICS_PATH, index=False)
        display(metrics_df)

        numeric_cols = ["json_valid", "field_coverage", "recommended_channel_match", "kpi_jaccard", "text_jaccard_vs_expected", "image_prompt_present", "latency_sec"]
        summary_df = metrics_df.groupby("model")[numeric_cols].mean(numeric_only=True).reset_index()
        display(summary_df)

        by_model = {}
        for item in all_outputs:
            by_model[(item["example_id"], item["model"])] = item

        table_rows = []
        for idx, record in enumerate(eval_records[:MAX_EVAL_EXAMPLES]):
            base_item = by_model.get((idx, "base"), {})
            ft_item = by_model.get((idx, "fine_tuned"), {})
            table_rows.append({
                "example_id": idx,
                "input": record["input"],
                "expected": json.dumps(record["output"], ensure_ascii=False),
                "base_generated": base_item.get("generated", ""),
                "fine_tuned_generated": ft_item.get("generated", ""),
            })
        comparison_df = pd.DataFrame(table_rows)
        comparison_df.to_csv(COMPARISON_TABLE_PATH, index=False)
        display(comparison_df)

        print("Metricas:", METRICS_PATH)
        print("Tabla comparativa:", COMPARISON_TABLE_PATH)
else:
    print("BUILD_COMPARISON_REPORT=False. Se omite reporte comparativo.")


## 12. Notas de lectura de resultados

Una mejora despues del fine-tuning deberia verse principalmente en:

- mas respuestas JSON validas;
- mayor cobertura de campos;
- `recommended_channel` mas alineado con el esperado;
- KPIs mas parecidos al dataset;
- `image_prompt` presente y util para integracion visual posterior;
- menor necesidad de reparacion manual de formato.

Estas metricas son indicativas. Para decidir calidad final, revisa tambien ejemplos lado a lado.
